# Gaps and the spectral window

The spectral window describes how the observation times redistribute power in
frequency. It is the Fourier transform of the sampling pattern: a continuous
run produces a narrow central peak, while gaps add sidelobes and usually
broaden the effective independent-frequency spacing.

This notebook is fully offline.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mimir import (
    TimeSeries,
    effective_frequency_spacing,
    spectral_window,
)

plt.rcParams["figure.figsize"] = (8, 4)


## Compare regular and gapped sampling

We create a 30-day time series at 30-minute cadence. The gapped version loses
a two-day block and a short interval every 5 days. The flux values are zeros
because the window depends only on the observation times.


In [ ]:
cadence_days = 30.0 / (24.0 * 60.0)
regular_time = np.arange(0.0, 30.0, cadence_days)

central_gap = (regular_time > 12.0) & (regular_time < 14.0)
periodic_gap = np.mod(regular_time, 5.0) < 0.25
gapped_time = regular_time[~(central_gap | periodic_gap)]

regular = TimeSeries(
    regular_time,
    np.zeros_like(regular_time),
    time_unit="d",
)
gapped = TimeSeries(
    gapped_time,
    np.zeros_like(gapped_time),
    time_unit="d",
)

{
    "regular duty cycle": regular.duty_cycle,
    "gapped duty cycle": gapped.duty_cycle,
    "regular samples": regular.n_samples,
    "gapped samples": gapped.n_samples,
}


In [ ]:
fig, ax = plt.subplots()
ax.plot(regular.time, np.ones_like(regular.time), "|", ms=10, label="Regular")
ax.plot(gapped.time, np.zeros_like(gapped.time), "|", ms=10, label="Gapped")
ax.set(
    xlabel="Time [d]",
    yticks=[],
    title="Observation times",
)
ax.legend()
plt.show()


`half_width` sets the integration range around zero frequency. It should be
wide enough to include the central peak and important sidelobes. Increasing
`oversampling` refines the numerical representation of the window.


In [ ]:
regular_window = spectral_window(
    time_series=regular,
    half_width=15.0,
    oversampling=20,
)
gapped_window = spectral_window(
    time_series=gapped,
    half_width=15.0,
    oversampling=20,
)

{
    "regular nominal spacing (uHz)": (
        regular_window.nominal_frequency_spacing
    ),
    "regular effective spacing (uHz)": (
        regular_window.effective_frequency_spacing
    ),
    "gapped nominal spacing (uHz)": (
        gapped_window.nominal_frequency_spacing
    ),
    "gapped effective spacing (uHz)": (
        gapped_window.effective_frequency_spacing
    ),
}


In [ ]:
fig, ax = plt.subplots()
ax.plot(
    regular_window.frequency,
    regular_window.power,
    label="Regular",
)
ax.plot(
    gapped_window.frequency,
    gapped_window.power,
    label="Gapped",
    alpha=0.85,
)
ax.set(
    xlabel=f"Frequency offset [{regular_window.frequency_unit}]",
    ylabel="Normalized window power",
    title="Sampling spectral windows",
    xlim=(-8.0, 8.0),
)
ax.legend()
plt.show()


The convenience function returns only the integrated result:


In [ ]:
spacing = effective_frequency_spacing(
    time_series=gapped,
    half_width=15.0,
    oversampling=20,
)
print(f"Effective spacing: {spacing:.4f} uHz")


## Interpreting the result

- `nominal_frequency_spacing` is \(1/T\), based only on the time span.
- `effective_frequency_spacing` is the area under the normalized window over
  the requested range.
- The effective value is useful when a likelihood or detection statistic
  needs an estimate of the number of approximately independent frequency bins.
- A larger integration range can capture distant sidelobes, so check
  convergence with respect to `half_width` for strongly gapped data.

The same calculation accepts a MAST-resolvable name:

```python
window = spectral_window(
    target="KIC 8006161",
    mast_kwargs={
        "search_kwargs": {"mission": "Kepler", "exptime": 60},
        "numax": 3500.0,
    },
)
```
